# Método para Obter a Matriz de Capacitâncias de uma Linha de Transmissão

A matriz de capacitâncias de uma linha de transmissão é obtida considerando os cabos como condutores carregados em relação ao solo e entre si. O método das imagens é usado para levar em conta o efeito do solo.

## Passos para Cálculo

### 1. Definição das Coordenadas dos Cabos
- **Coordenadas horizontais**: Representam as posições dos cabos no plano horizontal em relação a um eixo de referência (e.g., posição no espaçamento da torre).
- **Alturas médias**: Calculadas como a média aritmética entre as alturas dos cabos nas torres e no meio do vão.

### 2. Consideração da Capacitância Própria
A capacitância própria de um cabo leva em consideração sua altura em relação ao solo. O método das imagens cria um cabo imaginário simétrico abaixo do solo para considerar o efeito da terra.

A fórmula utilizada é:

$$
C_{ii} = \frac{2 \pi \varepsilon_0}{\ln\left(\frac{2h_i}{r}\right)}
$$

- $ \varepsilon_0 $: Permissividade do vácuo ($ 8.854 \times 10^{-12} \, F/m $).
- $ h_i $: Altura do cabo em relação ao solo.
- $ r $: Raio equivalente do cabo. Para simplificar, assumimos $ r = 1 \, \mu m $ para garantir que os valores do logaritmo sejam positivos.

### 3. Consideração da Capacitância Mútua
A capacitância mútua entre dois cabos depende da distância geométrica entre eles. A fórmula é:

$$
C_{ij} = -\frac{2 \pi \varepsilon_0}{\ln\left(\frac{d_{ij}}{r}\right)}
$$

- $ d_{ij} $: Distância geométrica entre os cabos $ i $ e $ j $, calculada como:
  $$
  d_{ij} = \sqrt{(x_i - x_j)^2 + (h_i - h_j)^2}
  $$

### 4. Montagem da Matriz de Capacitâncias
A matriz de capacitâncias $ C $ é composta por:
- Elementos diagonais ($ C_{ii} $): Capacitâncias próprias.
- Elementos fora da diagonal ($ C_{ij} $): Capacitâncias mútuas.

A matriz é simétrica, pois $ C_{ij} = C_{ji} $.

### Fórmulas Finalizadas
- Para a diagonal ($ i = j $):
  $$
  C_{ii} = \frac{2 \pi \varepsilon_0}{\ln\left(\frac{2h_i}{r}\right)}
  $$

- Para fora da diagonal ($ i \neq j $):
  $$
  C_{ij} = -\frac{2 \pi \varepsilon_0}{\ln\left(\frac{d_{ij}}{r}\right)}
  $$

## Aplicação no Código
No código Python:
1. Inicializamos uma matriz $ C $ de zeros.
2. Calculamos cada elemento da matriz usando as fórmulas acima.
3. Preenchemos os valores da matriz de capacitâncias.

Essa matriz pode ser usada em estudos de linhas de transmissão para determinar o comportamento elétrico do sistema.

### Observação
Certifique-se de ajustar os parâmetros (como o raio $ r $) para corresponder às características físicas reais dos cabos.


In [1]:
import numpy as np

# Function to calculate the capacitance matrix (4x4)
def calculate_capacitance_matrix(x_coords, h_avg, radii, epsilon_0=8.854e-12):
    n = len(x_coords)
    C = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                # Self-capacitance using the method of images
                h = h_avg[i]
                r = radii[i]
                C[i, j] = 2 * np.pi * epsilon_0 / np.log(2 * h / r)
            else:
                # Mutual capacitance between two conductors
                d_ij = np.sqrt((x_coords[i] - x_coords[j])**2 + (h_avg[i] - h_avg[j])**2)
                r_avg = np.sqrt(radii[i] * radii[j])
                C[i, j] = -2 * np.pi * epsilon_0 / np.log(d_ij / r_avg)
    return C

# Function to reduce the capacitance matrix (3x3)
def reduce_capacitance_matrix(C_full):
    # Extract submatrices
    C_ph_ph = C_full[:3, :3]       # 3x3 phases-to-phases
    C_ph_G = C_full[:3, 3:4]       # 3x1 phases-to-ground
    C_G_ph = C_full[3:4, :3]       # 1x3 ground-to-phases
    C_GG = C_full[3, 3]            # Scalar ground-to-ground

    # Reduced capacitance matrix for phases
    C_reduced = C_ph_ph - (C_ph_G @ C_G_ph) / C_GG
    return C_reduced

# Function to calculate sequence capacitances from reduced matrix
def calculate_sequence_capacitances(C_reduced):
    # Transformation matrix for phase to sequence components
    a = np.exp(2j * np.pi / 3)  # 120-degree rotation operator
    T = np.array([
        [1, 1, 1],
        [1, a, a**2],
        [1, a**2, a]
    ], dtype=complex) / 3

    # Sequence capacitance matrix
    C_seq = T @ C_reduced @ np.conjugate(T.T)
    return C_seq.real  # Only the real part is meaningful

# Example usage
# Horizontal coordinates (m)
x_coords = np.array([-1.1, 0.4, 1.1, 0.4])

# Heights at the tower and mid-span (m)
h_tower = np.array([10.0, 10.0, 10.0, 11.2])
h_midspan = np.array([8.0, 8.0, 8.0, 8.2])

# Average height of the cables
h_avg = (2 * h_tower + h_midspan) / 3

# Radii of the conductors (m)
phase_radius = 0.005  # 0.5 cm
ground_wire_radius = 0.005  # 0.5 cm
radii = np.array([phase_radius, phase_radius, phase_radius, ground_wire_radius])

# Step 1: Calculate full capacitance matrix
C_full = calculate_capacitance_matrix(x_coords, h_avg, radii)
print("Full Capacitance Matrix (4x4):")
print(C_full)

# Step 2: Calculate reduced capacitance matrix
C_reduced = reduce_capacitance_matrix(C_full)
print("\nReduced Capacitance Matrix (3x3):")
print(C_reduced)

# Step 3: Calculate sequence capacitances
C_sequence = calculate_sequence_capacitances(C_reduced)
print("\nSequence Capacitance Matrix:")
print(C_sequence)


Full Capacitance Matrix (4x4):
[[ 6.76363997e-12 -9.75340889e-12 -9.13970456e-12 -9.51319134e-12]
 [-9.75340889e-12  6.76363997e-12 -1.12576585e-11 -1.07912679e-11]
 [-9.13970456e-12 -1.12576585e-11  6.76363997e-12 -1.02900519e-11]
 [-9.51319134e-12 -1.07912679e-11 -1.02900519e-11  6.69140140e-12]]

Reduced Capacitance Matrix (3x3):
[[-6.76130107e-12 -2.50953964e-11 -2.37691113e-11]
 [-2.50953964e-11 -1.06395102e-11 -2.78524943e-11]
 [-2.37691113e-11 -2.78524943e-11 -9.06042477e-12]]

Sequence Capacitance Matrix:
[[-1.99883600e-11  7.23211869e-13  7.23211869e-13]
 [ 7.23211869e-13  5.58397399e-12 -4.16868268e-13]
 [ 7.23211869e-13 -4.16868268e-13  5.58397399e-12]]
